## 12-knn-v3-training.ipynb
Builds KNN model v3. Identical pipeline to v2 with one change:
`album_tags_matrix` is replaced by `album_genre_matrix` — a unified genre
feature that blends album tags (w=1.0), artist tags for sparsely-tagged albums
(w=0.5), and label tags (w=0.3), covering 11,247 tags vs the original 3,041.

Feature blocks:
- `album_genre_matrix`     (replaces album_tags_matrix)
- `album_labels_matrix`
- `album_types_matrix`
- `album_ratings_matrix`
- `album_country_matrix`
- `album_track_stats_matrix`

Artefacts saved to `../data/model_v3/`.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

os.makedirs('../data/features', exist_ok=True)
os.makedirs('../data/model_v3', exist_ok=True)

In [ ]:
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

X_genre       = load_npz('../data/features/album_genre_matrix.npz')
X_labels      = load_npz('../data/features/album_labels_matrix.npz')
X_types       = load_npz('../data/features/album_types_matrix.npz')
X_ratings     = load_npz('../data/features/album_ratings_matrix.npz')
X_country     = load_npz('../data/features/album_country_matrix.npz')
X_track_stats = load_npz('../data/features/album_track_stats_matrix.npz')

print('Feature blocks loaded:')
for name, X in [('X_genre', X_genre), ('X_labels', X_labels), ('X_types', X_types),
                ('X_ratings', X_ratings), ('X_country', X_country), ('X_track_stats', X_track_stats)]:
    print(f'  {name:<18} {str(X.shape):<25} nnz={X.nnz:,}')

In [ ]:
# Expand all matrices to the full album universe
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

all_blocks = {
    'X_genre':       X_genre,
    'X_labels':      X_labels,
    'X_types':       X_types,
    'X_ratings':     X_ratings,
    'X_country':     X_country,
    'X_track_stats': X_track_stats,
}

if len(album_id_order) < len(full_album_ids):
    print(f'Expanding {len(album_id_order):,} → {len(full_album_ids):,} albums...')
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    all_blocks = {k: _expand(v, current_pos, n_full) for k, v in all_blocks.items()}
    album_id_order = full_album_ids.tolist()

X_genre, X_labels, X_types, X_ratings, X_country, X_track_stats = all_blocks.values()

In [ ]:
# Block weights — same as v2 except genre replaces tags
W_GENRE       = 1.0
W_LABELS      = 1.0
W_TYPES       = 1.0
W_RATINGS     = 1.0
W_COUNTRY     = 0.2   # tune: 0.1 – 0.5
W_TRACK_STATS = 1.0

X_final_v3 = hstack([
    X_genre       * W_GENRE,
    X_labels      * W_LABELS,
    X_types       * W_TYPES,
    X_ratings     * W_RATINGS,
    X_country     * W_COUNTRY,
    X_track_stats * W_TRACK_STATS,
]).tocsr()

print(f'X_final_v3: {X_final_v3.shape[0]:,} albums x {X_final_v3.shape[1]:,} features  (nnz={X_final_v3.nnz:,})')
print(f'\nFeature block summary:')
print(f'  genre        w={W_GENRE}   : {X_genre.shape[1]:,} cols')
print(f'  labels       w={W_LABELS}   : {X_labels.shape[1]:,} cols')
print(f'  types        w={W_TYPES}   : {X_types.shape[1]:,} cols')
print(f'  ratings      w={W_RATINGS}   : {X_ratings.shape[1]:,} cols')
print(f'  country      w={W_COUNTRY} : {X_country.shape[1]:,} cols')
print(f'  track_stats  w={W_TRACK_STATS}   : {X_track_stats.shape[1]:,} cols')
print(f'  Total                   : {X_final_v3.shape[1]:,} cols')

In [ ]:
# Safe column pruning
col_nnz      = np.diff(X_final_v3.tocsc().indptr)
row_lengths  = np.diff(X_final_v3.indptr)
has_features = row_lengths > 0

col_nnz_vals          = col_nnz[X_final_v3.indices]
nonempty_rows         = np.where(has_features)[0]
row_starts            = X_final_v3.indptr[nonempty_rows]
max_col_nnz_per_album = np.zeros(X_final_v3.shape[0], dtype=col_nnz.dtype)
max_col_nnz_per_album[nonempty_rows] = np.maximum.reduceat(col_nnz_vals, row_starts)

safe_threshold = int(max_col_nnz_per_album[has_features].min())
keep_cols      = col_nnz >= safe_threshold
X_knn_v3       = X_final_v3[:, keep_cols]

print(f'Safe threshold : {safe_threshold}')
print(f'Columns before : {X_final_v3.shape[1]:,}')
print(f'Columns after  : {X_knn_v3.shape[1]:,}  ({keep_cols.mean()*100:.1f}% retained)')
print(f'Albums with features: {has_features.sum():,}  ({has_features.mean()*100:.1f}%)')

In [ ]:
X_knn_annotated_v3    = X_knn_v3[has_features].copy()
album_ids_annotated_v3 = np.array(album_id_order)[has_features]

nan_count = np.isnan(X_knn_annotated_v3.data).sum()
if nan_count:
    print(f'Removing {nan_count:,} NaN entries...')
    np.nan_to_num(X_knn_annotated_v3.data, nan=0.0, copy=False)
    X_knn_annotated_v3.eliminate_zeros()

X_knn_norm_v3 = normalize(X_knn_annotated_v3, norm='l2')
print(f'Fitting on {X_knn_norm_v3.shape[0]:,} albums x {X_knn_norm_v3.shape[1]:,} features')

In [ ]:
model_v3 = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model_v3.fit(X_knn_norm_v3)
print('Model v3 fitted.')

In [ ]:
distances, indices = model_v3.kneighbors(X_knn_norm_v3[0], n_neighbors=11)

print(f'Query album id: {album_ids_annotated_v3[0]}')
print(f"\n{'rank':<6} {'album_id':<40} {'cosine distance':>15}")
print('-' * 62)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = '(query)' if rank == 0 else ''
    print(f"{rank:<6} {str(album_ids_annotated_v3[idx]):<40} {dist:>15.4f}  {label}")

In [ ]:
joblib.dump(model_v3,                   '../data/model_v3/knn_model_v3.joblib')
save_npz('../data/model_v3/X_knn_norm_v3.npz', X_knn_norm_v3)
np.save('../data/model_v3/album_ids_annotated_v3.npy', album_ids_annotated_v3)
np.save('../data/model_v3/has_features_v3.npy',        has_features)

print('Saved:')
print('  ../data/model_v3/knn_model_v3.joblib')
print('  ../data/model_v3/X_knn_norm_v3.npz')
print('  ../data/model_v3/album_ids_annotated_v3.npy')
print('  ../data/model_v3/has_features_v3.npy')